# 02. MEG Emotion Recognition

Generate a synthetic MEG feature dataset when it is missing, train an SVM classifier, and save backend-compatible artifacts.


In [1]:
from pathlib import Path
import sys

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "NeuroSense" / "webdev" / "backend").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root for this notebook.")

PROJECT_ROOT = _find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "NeuroSense" / "notebooks"
BACKEND_DIR = PROJECT_ROOT / "NeuroSense" / "webdev" / "backend"

for path in (NOTEBOOKS_DIR, BACKEND_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from notebook_support import bootstrap_notebook

ctx = bootstrap_notebook(PROJECT_ROOT)
DATASETS_DIR = ctx["datasets_dir"]
ARTIFACTS_DIR = ctx["artifacts_dir"]
CACHE_DIR = ctx["cache_dir"]
RANDOM_STATE = ctx["random_state"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Datasets directory: {DATASETS_DIR}")
print(f"Artifacts directory: {ARTIFACTS_DIR}")


Project root: /Users/devashishsingh/Desktop/human emotion recognition system
Datasets directory: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/datasets
Artifacts directory: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/artifacts


In [2]:
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.svm import SVC

meg_path = DATASETS_DIR / "meg" / "meg_features.csv"
meg_path.parent.mkdir(parents=True, exist_ok=True)

if not meg_path.exists():
    rng = np.random.default_rng(RANDOM_STATE)
    n_samples = 1000
    features = rng.normal(size=(n_samples, 50))
    labels = rng.choice(["POSITIVE", "NEGATIVE", "NEUTRAL"], size=n_samples, p=[0.34, 0.33, 0.33])

    for index, label in enumerate(labels):
        if label == "POSITIVE":
            features[index, :10] += 1.5
        elif label == "NEGATIVE":
            features[index, 10:20] += 1.5
        else:
            features[index, 20:30] += 0.5

    meg_df = pd.DataFrame(features, columns=[f"meg_f{idx}" for idx in range(features.shape[1])])
    meg_df["label"] = labels
    meg_df.to_csv(meg_path, index=False)
    print(f"Generated synthetic MEG dataset at {meg_path}")

df = pd.read_csv(meg_path)
X = df.drop(columns=["label"]).astype(np.float32)
y = df["label"].astype(str)

print("Dataset shape:", df.shape)
print("Label counts:", y.value_counts().to_dict())


Dataset shape: (600, 51)
Label counts: {'NEGATIVE': 207, 'POSITIVE': 197, 'NEUTRAL': 196}


In [3]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X.values,
    y_enc,
    test_size=0.2,
    stratify=y_enc,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

model = SVC(kernel="rbf", C=10.0, probability=True, random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(
    make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10.0, probability=True, random_state=RANDOM_STATE)),
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)

print(f"MEG test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"MEG CV mean: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(classification_report(y_test, y_pred, target_names=le.classes_))


MEG test accuracy: 0.9917
MEG CV mean: 0.9979 +/- 0.0042
              precision    recall  f1-score   support

    NEGATIVE       1.00      1.00      1.00        42
     NEUTRAL       1.00      0.97      0.99        39
    POSITIVE       0.97      1.00      0.99        39

    accuracy                           0.99       120
   macro avg       0.99      0.99      0.99       120
weighted avg       0.99      0.99      0.99       120



In [4]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("MEG confusion matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

y_test_bin = label_binarize(y_test, classes=list(range(len(le.classes_))))
y_score = model.predict_proba(X_test_sc)

plt.figure(figsize=(7, 5))
for index, class_name in enumerate(le.classes_):
    fpr, tpr, _ = roc_curve(y_test_bin[:, index], y_score[:, index])
    plt.plot(fpr, tpr, label=f"{class_name} (AUC={auc(fpr, tpr):.2f})")
plt.plot([0, 1], [0, 1], "k--")
plt.title("MEG one-vs-rest ROC curves")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.legend()
plt.tight_layout()
plt.show()


/var/folders/y1/kwl357md29bd8ggvss23h_yc0000gn/T/ipykernel_59686/1619152407.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/y1/kwl357md29bd8ggvss23h_yc0000gn/T/ipykernel_59686/1619152407.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
artifact_dir = ARTIFACTS_DIR / "meg"
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, artifact_dir / "meg_model.pkl")
joblib.dump(scaler, artifact_dir / "meg_scaler.pkl")
joblib.dump(le, artifact_dir / "meg_label_encoder.pkl")

print("Saved MEG artifacts to:", artifact_dir)


Saved MEG artifacts to: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/artifacts/meg
